# 01 - Data Quality Assessment

This notebook documents the source-data validation performed before
the final churn analysis.

It focuses on dataset structure, duplicates, missing values, numeric
conversion, category validation, and the known TotalCharges issue.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SOURCE = ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"

df = pd.read_csv(SOURCE)

df.shape


## Dataset Structure

The source is expected to contain 7,043 customer rows and 21 columns.


In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())


## Duplicate Checks


In [ ]:
print(
    "Duplicate customer IDs:",
    df["customerID"].duplicated().sum(),
)

print(
    "Duplicate complete rows:",
    df.duplicated().sum(),
)


## Missing-Value Review

The original blank TotalCharges records are stored as whitespace,
which means a normal null-value check alone does not identify them.


In [ ]:
df.isna().sum()


## Numeric Validation


In [ ]:
for column in [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]:
    converted = pd.to_numeric(
        df[column],
        errors="coerce",
    )

    print(
        column,
        "invalid/missing after conversion:",
        converted.isna().sum(),
    )


## TotalCharges Investigation


In [ ]:
total_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce",
)

invalid = df.loc[
    total_numeric.isna()
].copy()

print(
    "Invalid TotalCharges:",
    len(invalid),
)

print(
    "Zero-tenure customers:",
    (df["tenure"] == 0).sum(),
)

print(
    "Invalid records with zero tenure:",
    (invalid["tenure"] == 0).sum(),
)

print("\nOriginal values:")
print(
    invalid["TotalCharges"]
    .map(repr)
    .value_counts()
)

print("\nChurn:")
print(
    invalid["Churn"]
    .value_counts()
)


## Core Category Review


In [ ]:
for column in [
    "Churn",
    "Contract",
    "InternetService",
    "PaymentMethod",
]:
    print(
        f"\n{column}"
    )

    print(
        df[column]
        .value_counts(
            dropna=False
        )
    )


## Data-Quality Decision

The 11 blank TotalCharges values belong to the same 11 customers with
zero months of tenure. The customer records are retained and the
TotalCharges values are treated as missing numeric values.

No customer records are removed for the project's primary segment
analysis.
